# Claimback: train everything in one notebook
**Kaggle settings (right panel → Session options):** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = **Files only** (if shown).
**Secret:** Add-ons → Secrets → `HF_TOKEN` (Hugging Face *Write* token), switched on for this notebook.

**How to use (follow the guide):**
1. Tonight, run every cell from top to bottom with `QUICK_TEST = True`. Data prep and the baseline are real; training is a tiny 5-minute rehearsal.
2. Download the baseline files for your teammate (Step 11).
3. Set `QUICK_TEST = False` → **Save Version → Save & Run All (Commit)**. It reruns everything on Kaggle's servers (about 2–4 hours) while you sleep.

## Step 1: Settings (the only cell you edit)

In [ ]:
HF_USERNAME = "YOUR_HF_USERNAME"       # <-- your Hugging Face username (top right of huggingface.co)
QUICK_TEST  = True                     # True = 5-minute rehearsal of training. False = the real overnight run.
MODEL       = "roberta-base"           # safe default. Optional later upgrade: "microsoft/deberta-v3-base"
MAX_LEN     = 384                      # how many word-pieces of each complaint the model reads. 256 = faster
EPOCHS_WIN, EPOCHS_ISSUE = 2, 1
VERSION     = "v1"
ISSUE_GROUPS = ["Account problems", "Card or transaction dispute", "Debt collection",
                "Fees and charges", "Loan or mortgage servicing", "Scam or unauthorised payment"]
import os, re, time, json, zipfile, requests, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", message=".*infer format.*")   # harmless date-format message
print("settings ok | QUICK_TEST =", QUICK_TEST)

## Step 2: The data links
In Aug 2026 the CFPB stopped putting the written stories in its normal download. The stories now live in a
**narratives archive** of zip files. These are the exact links from that archive page (checked 14 Nov 2026).
We skip the oldest file (Dec 2011 to Apr 2018): we don't need it, and it saves time and disk.

In [ ]:
BASE = "https://files.consumerfinance.gov/f/documents/"
NAMES = ["21_August_2026", "20_July_2026", "19_June_2026", "18_May_2026", "17_April_2026", "16_March_2026",
         "15_January_2026_through_February_2026", "14_November_2025_through_December_2025",
         "13_September_2025_through_October_2025", "12_July_2025_through_August_2025",
         "11_May_2025_through_June_2025", "10_March_2025_through_April_2025",
         "9_January_2025_through_February_2025", "8_November_2024_through_December_2024",
         "7_August_2024_through_October_2024", "6_April_2024_through_July_2024",
         "5_September_2023_through_March_2024", "4_November_2022_through_August_2023",
         "3_May_2021_through_October_2022", "2_May_2018_through_April_2021"]
links = [BASE + "CCDB_Export_" + n + ".zip" for n in NAMES]
print(len(links), "files")      # should print: 20 files

## Step 3: Download and check every file (10–30 minutes)
Each file must print `status: 200` and `valid zip: True`. Files already downloaded and valid are skipped,
so if anything fails you can simply run this cell again.

In [ ]:
ARCH = "/kaggle/temp/archive" if os.path.isdir("/kaggle/temp") else "archive"   # big temporary files, not saved
os.makedirs(ARCH, exist_ok=True)
failed = []
for l in links:
    p = ARCH + "/" + l.split("/")[-1]
    if os.path.exists(p) and zipfile.is_zipfile(p):
        print("already ok:", p); continue
    print("downloading", p)
    try:
        r = requests.get(l, headers={"User-Agent": "Mozilla/5.0"}, stream=True, timeout=900)
        print("   status:", r.status_code, "| type:", r.headers.get("Content-Type"))
        with open(p, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
    except Exception as e:
        print("   FAILED:", e)
    ok = os.path.exists(p) and zipfile.is_zipfile(p)
    print("   valid zip:", ok, "|", round(os.path.getsize(p) / 1e6, 1) if os.path.exists(p) else 0, "MB")
    if not ok:
        if os.path.exists(p):
            print("   file starts with:", open(p, "rb").read(80)); os.remove(p)   # remove the broken file
        failed.append(l)
print("\nFAILED:", len(failed), "of", len(links))
for l in failed: print("  ", l)
print("Next: run this cell again once. Still failing? See the guide, 'If a download fails'." if failed else "All files downloaded and valid!")

## Step 3b: Only if you uploaded a file yourself (see the guide, problem D1)
If a download kept failing and you uploaded that zip to Kaggle as a Dataset, this copies it in.
Kaggle sometimes unzips uploads, so a .csv is zipped back up. If you uploaded nothing, it just says so.

In [ ]:
import glob, shutil
found = 0
for f in glob.glob("/kaggle/input/**/*", recursive=True):
    name = os.path.basename(f)
    if f.lower().endswith(".zip") and zipfile.is_zipfile(f):
        shutil.copy(f, ARCH + "/" + name); found += 1; print("copied", f)
    elif f.lower().endswith(".csv") and "CCDB" in name:
        with zipfile.ZipFile(ARCH + "/" + name[:-4] + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(f, name)
        found += 1; print("zipped and copied", f)
print(found, "uploaded file(s) added" if found else "- no uploaded files (that's normal)")

## Step 4: Look inside the zips
Expect `missing: nothing` (or only `Complaint ID` / `Sub-issue`, which are optional) for each file.

In [ ]:
def norm(c): return re.sub(r"[^a-z]", "", str(c).lower())
WANT = {"Date received": ["datereceived"], "Product": ["product"], "Sub-product": ["subproduct"],
        "Issue": ["issue"], "Sub-issue": ["subissue"], "Company": ["company"],
        "Company response to consumer": ["companyresponsetoconsumer"], "Complaint ID": ["complaintid"],
        "Consumer complaint narrative": ["consumercomplaintnarrative", "complaintnarrative", "narrative", "complainttext"]}
NEEDED = ["Date received", "Product", "Issue", "Company", "Company response to consumer", "Consumer complaint narrative"]
def column_map(cols):
    n = {norm(c): c for c in cols}
    m = {want: next((n[o] for o in opts if o in n), None) for want, opts in WANT.items()}
    if m["Consumer complaint narrative"] is None:
        m["Consumer complaint narrative"] = next((c for c in cols if "narrative" in str(c).lower()), None)
    return m
FILES = []
for f in sorted(os.listdir(ARCH)):
    if not zipfile.is_zipfile(ARCH + "/" + f):
        print("SKIP (not a valid zip, re-run Step 3):", f); continue
    z = zipfile.ZipFile(ARCH + "/" + f)
    csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
    if not csvs:
        print("NO CSV INSIDE", f, "contains:", z.namelist()[:5]); continue
    for inner in csvs:
        with z.open(inner) as fh:
            cols = list(pd.read_csv(fh, nrows=0).columns)
        m = column_map(cols)
        missing = [k for k, v in m.items() if v is None]
        print(f, "| missing:", missing or "nothing")
        if any(k in missing for k in NEEDED):
            print("   columns in this file:", cols)
        FILES.append((f, inner, m))
print(len(FILES), "files ready")

## Step 5: Read only what we need (15–40 minutes)
Keeps complaints that have a story, are closed, and aren't about credit reports (those are rarely about money).

In [ ]:
keep, t0, bad_dates = [], time.time(), 0
for f, inner, m in FILES:
    if any(m[k] is None for k in NEEDED):
        print("SKIPPING", f, "- a needed column is missing"); continue
    use = {v: k for k, v in m.items() if v is not None}
    with zipfile.ZipFile(ARCH + "/" + f).open(inner) as fh:
        for ch in pd.read_csv(fh, usecols=list(use), chunksize=200_000, low_memory=False,
                              dtype=str, on_bad_lines="skip"):
            ch = ch.rename(columns=use)
            for c in WANT:
                if c not in ch:
                    ch[c] = None
            ch = ch.dropna(subset=["Consumer complaint narrative"])
            ch = ch[ch["Consumer complaint narrative"].str.len() > 30]
            ch = ch[ch["Company response to consumer"].fillna("").str.startswith("Closed")]
            ch = ch[~ch["Product"].fillna("").str.contains("credit report", case=False)]
            ch["date"] = pd.to_datetime(ch["Date received"], errors="coerce")   # per chunk: each file keeps its own date format
            bad_dates += int(ch["date"].isna().sum())
            keep.append(ch[list(WANT) + ["date"]])
    print(f"{f}: done  ({time.time()-t0:.0f}s, {sum(map(len, keep)):,} rows so far, {bad_dates} unreadable dates)")
assert keep, "No data was read. Go back and run Step 3 (download) and Step 4 again; Step 4 must say 'files ready' with a number above 0."
df = pd.concat(keep, ignore_index=True); del keep
if df["Complaint ID"].notna().any():
    df = df.drop_duplicates("Complaint ID")
df["label"] = (df["Company response to consumer"].str.strip().str.lower() == "closed with monetary relief").astype(int)
df = df.dropna(subset=["date"])
print("\nrows:", f"{len(df):,}", "| money-back rate:", round(df.label.mean(), 3), "| dates:", df.date.min().date(), "to", df.date.max().date())
print(df["Company response to consumer"].value_counts())

## Step 6: Sort every complaint into one of the 6 issue groups
If `None` is a big group, it just means some products don't fit; they are dropped.

In [ ]:
def group(product, issue_text):
    p, i = str(product).lower(), str(issue_text).lower()
    if re.search(r"fraud|scam|unauthori|not authorized|didn.t authorize|identity theft|stolen", i):
        return "Scam or unauthorised payment"
    if "debt collection" in p:                                         return "Debt collection"
    if re.search(r"mortgage|loan|lease", p):                           return "Loan or mortgage servicing"
    if re.search(r"fee|interest|charge", i):                           return "Fees and charges"
    if re.search(r"dispute|purchase shown|transaction|billing|statement", i): return "Card or transaction dispute"
    if re.search(r"checking|savings|bank account|money transfer|prepaid|credit card|virtual currency|money service", p):
        return "Account problems"
    return None
df["issue_group"] = [group(p, f"{a} {b}") for p, a, b in zip(df["Product"], df["Issue"], df["Sub-issue"])]
print(df["issue_group"].value_counts(dropna=False))
df = df.dropna(subset=["issue_group"])
print("\nmoney-back rate per group:"); print(df.groupby("issue_group").label.mean().round(3))

## Step 7: Read 3 random complaints (always look at your data)

In [ ]:
for _, r in df.sample(3, random_state=1).iterrows():
    print(r.issue_group, "| money back:", r.label, "|", r.Company)
    print(r["Consumer complaint narrative"][:600]); print("-" * 80)

## Step 8: Split by time, balance, hide company names
- **train** = before 2024 · **val** = 2024 · **test** = 2025 onwards (the model never sees the future)
- Training set: money-back complaints + twice as many others (max about 80,000 rows)
- Val and test keep their natural ratio (max 20,000 each), so the scores are honest

In [ ]:
RNG = 42
tr_all = df[df.date < "2024-01-01"]
va_all = df[(df.date >= "2024-01-01") & (df.date < "2025-01-01")]
te_all = df[df.date >= "2025-01-01"]
print("available:", len(tr_all), len(va_all), len(te_all))
pos = tr_all[tr_all.label == 1]; pos = pos.sample(min(len(pos), 27_000), random_state=RNG)
neg_all = tr_all[tr_all.label == 0]; neg = neg_all.sample(min(len(neg_all), 2 * len(pos)), random_state=RNG)
train = pd.concat([pos, neg]).sample(frac=1, random_state=RNG)
val  = va_all.sample(min(len(va_all), 20_000), random_state=RNG)
test = te_all.sample(min(len(te_all), 20_000), random_state=RNG)

BRANDS = ["chase","jpmorgan","wells fargo","bank of america","citi","citibank","capital one","discover","synchrony",
          "td bank","pnc","us bank","u.s. bank","navy federal","paypal","venmo","cash app","cashapp","zelle","coinbase",
          "american express","amex","barclays","ally","usaa","regions","truist","suntrust","santander","comenity",
          "credit one","fifth third","huntington","citizens","keybank","chime","varo","sofi","goldman","apple card"]
top = df["Company"].value_counts().head(150).index.str.lower().tolist()
names = sorted({n.strip() for n in BRANDS + [re.sub(r"[,.]| inc$| n a$| llc$| corporation$| corp$| & co$| co$", "", t) for t in top]},
               key=len, reverse=True)
names = [n for n in names if len(n) >= 4 or n in ("citi", "amex", "pnc", "usaa")]
PAT = re.compile(r"\b(" + "|".join(re.escape(n) for n in names) + r")\b", re.I)
for d in (train, val, test):
    d["text"] = d["Consumer complaint narrative"].map(lambda t: PAT.sub("[COMPANY]", t))
    d["company"] = d["Company"]
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, len(d), "money-back rate", round(d.label.mean(), 3))

## Step 9: Save the data files

In [ ]:
COLS = ["text", "label", "issue_group", "company", "date"]
train[COLS].to_parquet("train.parquet"); val[COLS].to_parquet("val.parquet"); test[COLS].to_parquet("test.parquet")
br = df.groupby(["Company", "issue_group"]).label.agg(["mean", "count"]).reset_index()
br.columns = ["company", "issue_group", "rate", "n"]
br[br.n >= 30].to_csv("base_rates.csv", index=False)
test[test.label == 1][["text", "issue_group"]].head(400).to_csv("uplift_candidates.csv", index=False)
del df, tr_all, va_all, te_all                # free memory for training
import shutil
if ARCH == "archive": shutil.rmtree(ARCH)    # only if the zips landed in the saved folder (keeps the output small)
print(os.listdir("."))

## Step 10: Baseline model (TF-IDF + logistic regression, about 5 minutes)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
import joblib
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=100_000, sublinear_tf=True)
Xtr, Xva, Xte = vec.fit_transform(train.text), vec.transform(val.text), vec.transform(test.text)
win = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, train.label)
iss = LogisticRegression(max_iter=2000).fit(Xtr, train.issue_group)
BASE_M = {"val_auc": roc_auc_score(val.label, win.predict_proba(Xva)[:, 1]),
          "test_auc": roc_auc_score(test.label, win.predict_proba(Xte)[:, 1]),
          "test_issue_macro_f1": f1_score(test.issue_group, iss.predict(Xte), average="macro")}
print(BASE_M); json.dump(BASE_M, open("baseline_metrics.json", "w"), indent=1)
joblib.dump({"vectorizer": vec, "win_model": win, "issue_model": iss}, "tfidf.joblib", compress=3)
print("tfidf.joblib size:", round(os.path.getsize("tfidf.joblib") / 1e6, 1), "MB")

## Step 11: Download the baseline files for your teammate (tonight only)
In the right panel, open **Output** (`/kaggle/working`). For each of `tfidf.joblib`, `base_rates.csv`,
`uplift_candidates.csv` and `baseline_metrics.json`: hover → **⋮** → **Download**. Then carry on.

## Step 12: Log in to Hugging Face and prepare the training data

In [ ]:
import torch, inspect
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
if HF_USERNAME == "YOUR_HF_USERNAME":
    print("WARNING: you haven't put your Hugging Face username in Step 1, so the upload to Hugging Face will be skipped.")
try:
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("logged in to Hugging Face")
except Exception as e:
    print("HUGGING FACE LOGIN FAILED:", e)
    print("Fix: Add-ons -> Secrets -> make sure HF_TOKEN exists and its switch is ON for this notebook, then run this cell again.")
    raise
if QUICK_TEST:
    train_m, val_m, test_m = (d.sample(min(len(d), n), random_state=0) for d, n in ((train, 2000), (val, 1000), (test, 1000)))
    VERSION = "test"
else:
    train_m, val_m, test_m = train, val, test
print("GPUs:", torch.cuda.device_count(), "| rows:", len(train_m), len(val_m), len(test_m), "| version:", VERSION)

## Step 13: Turn text into numbers the model understands (tokenise)

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
tok = AutoTokenizer.from_pretrained(MODEL)
def make(d, task):
    x = pd.DataFrame({"text": d.text.values,
                      "labels": d.label.values if task == "win" else d.issue_group.map(ISSUE_GROUPS.index).values})
    return Dataset.from_pandas(x, preserve_index=False).map(
        lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])
data = {t: {"train": make(train_m, t), "val": make(val_m, t), "test": make(test_m, t)} for t in ("win", "issue")}
print(data["win"]["train"])

## Step 14: Training helper (works with old and new library versions)

In [ ]:
def win_metrics(p):
    pr = torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy()
    return {"auc": roc_auc_score(p.label_ids, pr), "f1": f1_score(p.label_ids, pr > 0.5)}
def issue_metrics(p):
    return {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}
STEPS = 50 if QUICK_TEST else 1000
def train_one(task, n_labels, metrics, best, epochs):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=n_labels)
    a = dict(output_dir=f"out_{task}", per_device_train_batch_size=16, per_device_eval_batch_size=64,
             learning_rate=2e-5, num_train_epochs=epochs, weight_decay=0.01,
             fp16=torch.cuda.is_available(), save_strategy="steps", save_steps=STEPS, eval_steps=STEPS,
             save_total_limit=2, load_best_model_at_end=True, metric_for_best_model=best,
             logging_steps=50, report_to="none")
    params = inspect.signature(TrainingArguments).parameters
    a["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "steps"
    if "warmup_ratio" in params: a["warmup_ratio"] = 0.06     # older library versions
    else:                        a["warmup_steps"] = 0.06     # newer versions: a number below 1 means "6% of steps"
    tk = {"processing_class": tok} if "processing_class" in inspect.signature(Trainer).parameters else {"tokenizer": tok}
    trainer = Trainer(model=model, args=TrainingArguments(**a), train_dataset=data[task]["train"],
                      eval_dataset=data[task]["val"], data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=metrics, **tk)
    trainer.train()
    return trainer
print("ready")

## Step 15: Train the money-back model (real run: about 1–3 hours)

In [ ]:
win_trainer = train_one("win", 2, win_metrics, "auc", EPOCHS_WIN)
win_trainer.save_model("model_win"); tok.save_pretrained("model_win")
try:
    win_trainer.model.push_to_hub(f"{HF_USERNAME}/claimback-win-{VERSION}")
    tok.push_to_hub(f"{HF_USERNAME}/claimback-win-{VERSION}")
    print("pushed to Hugging Face")
except Exception as e:
    print("PUSH FAILED (model is still saved in the model_win folder of the output):", e)

## Step 16: Train the issue-type model (real run: about 30–90 minutes)

In [ ]:
issue_trainer = train_one("issue", len(ISSUE_GROUPS), issue_metrics, "macro_f1", EPOCHS_ISSUE)
issue_trainer.save_model("model_issue"); tok.save_pretrained("model_issue")
try:
    issue_trainer.model.push_to_hub(f"{HF_USERNAME}/claimback-issue-{VERSION}")
    tok.push_to_hub(f"{HF_USERNAME}/claimback-issue-{VERSION}")
    print("pushed to Hugging Face")
except Exception as e:
    print("PUSH FAILED (model is still saved in the model_issue folder of the output):", e)

## Step 17: Evaluate on the test set, calibrate, make charts

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
def probs(trainer, ds): return torch.softmax(torch.tensor(trainer.predict(ds).predictions), -1).numpy()
pv = probs(win_trainer, data["win"]["val"])[:, 1]
pt = probs(win_trainer, data["win"]["test"])[:, 1]
cal = IsotonicRegression(out_of_bounds="clip").fit(pv, val_m.label.values)   # fitted on VAL only
joblib.dump(cal, "calibrator.joblib")
pt_cal = cal.predict(pt)
yt = test_m.label.values
issue_pred = probs(issue_trainer, data["issue"]["test"]).argmax(-1)
M = {"model": MODEL, "version": VERSION, "quick_test": QUICK_TEST, "n_train": len(train_m), "n_test": len(test_m),
     "test_auc": float(roc_auc_score(yt, pt)), "test_f1": float(f1_score(yt, pt > 0.5)),
     "baseline_test_auc": float(BASE_M["test_auc"]),
     "test_issue_macro_f1": float(f1_score(test_m.issue_group.map(ISSUE_GROUPS.index), issue_pred, average="macro")),
     "baseline_issue_macro_f1": float(BASE_M["test_issue_macro_f1"]),
     "test_money_back_rate": float(yt.mean())}
M["auc_by_issue"] = {}
for g in ISSUE_GROUPS:
    msk = (test_m.issue_group == g).values
    if len(set(yt[msk])) == 2:
        M["auc_by_issue"][g] = float(roc_auc_score(yt[msk], pt[msk]))
seen = train.company.value_counts()
rare = (~test_m.company.isin(seen[seen >= 50].index)).values
if len(set(yt[rare])) == 2:
    M["auc_rare_companies"] = float(roc_auc_score(yt[rare], pt[rare])); M["n_rare"] = int(rare.sum())
json.dump(M, open("metrics.json", "w"), indent=1); print(json.dumps(M, indent=1))

fig, ax = plt.subplots(figsize=(5, 5))
for p, name in [(pt, "raw"), (pt_cal, "calibrated")]:
    y, x = calibration_curve(yt, p, n_bins=10, strategy="quantile"); ax.plot(x, y, "o-", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("predicted chance"); ax.set_ylabel("actual money-back rate")
ax.legend(); ax.set_title("Calibration (test set)"); fig.savefig("calibration.png", dpi=200, bbox_inches="tight")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["Random guess", "TF-IDF baseline", "Fine-tuned " + MODEL.split("/")[-1]],
       [0.5, M["baseline_test_auc"], M["test_auc"]], color=["#bbbbbb", "#999999", "#1f6feb"])
ax.set_ylim(0.4, 1); ax.set_ylabel("AUC on 2025+ complaints"); fig.savefig("auc_vs_baseline.png", dpi=200, bbox_inches="tight")
import shutil
for d in ("out_win", "out_issue"):
    shutil.rmtree(d, ignore_errors=True)      # checkpoints not needed; keeps the output small
print("ALL DONE")

## Finished
**Quick test:** if you reached "ALL DONE", everything works. Go back to Step 1, set `QUICK_TEST = False`, and click
**Save Version → Save & Run All (Commit)**.

**After the real run:** open the version → **Output** and download `metrics.json`, `calibrator.joblib`,
`calibration.png` and `auc_vs_baseline.png`. Your models are on Hugging Face as `claimback-win-v1` and
`claimback-issue-v1` (make them Public there).